# Simpsons Character Classification

This project classifies characters from *The Simpsons* using transfer learning with PyTorch. The goal is to build a practical image-classification pipeline: dataset loading, stratified validation, class-imbalance handling, model fine-tuning, error analysis, and Kaggle-style submission generation.

The repository is structured as a small portfolio project: the notebook explains the experiment end to end, while reusable training, visualization, and label-audit code lives in separate Python modules.

## What this demonstrates

- Transfer learning with a pretrained computer-vision model.
- Clean train/validation split with stratification.
- Handling class imbalance with weighted sampling and weighted loss.
- Reusable project modules instead of notebook-only helper code.
- Error analysis workflow for identifying suspicious labels.
- Reproducible Colab setup with GitHub-hosted code and dataset release assets.

The public test set contains 991 images. The model predicts one character label per image and writes the final predictions to a submission CSV.

## 1. Environment Setup

This section imports the core libraries, verifies GPU availability, and mounts Google Drive for persistent artifacts such as checkpoints, audit tables, and the cached dataset archive.

In [ ]:
# we will verify that GPU is enabled for this notebook
# following should print: CUDA is available!  Training on GPU ...
#
# if it prints otherwise, then you need to enable GPU:
# from Menu > Runtime > Change Runtime Type > Hardware Accelerator > GPU

import torch
import numpy as np

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')


from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
%matplotlib inline

## 2. Project Configuration

The notebook loads source code from GitHub and stores only large runtime artifacts on Google Drive. This keeps the repository lightweight while making Colab runs reproducible.

In [ ]:
# Training device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset paths after extracting the archive in Colab
TRAIN_DIR = Path("/content/train")
TEST_DIR = Path("/content/testset")
SAMPLE_SUBMISSION_PATH = Path("/content/sample_submission.csv")

# Load project code from GitHub. Use Drive only for large runtime artifacts.
import subprocess
import sys

REPO_URL = "https://github.com/eritry/simpsons-classification.git"
CODE_DIR = Path("/content/simpsons-classification")

if (CODE_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
elif not (CODE_DIR / "training.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)], check=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Single Google Drive root for persistent artifacts.
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/simpsons")
CHECKPOINT_DIR = DRIVE_ARTIFACT_DIR / "checkpoints"
LABEL_AUDIT_DIR = DRIVE_ARTIFACT_DIR / "label_audit"
DATASET_DIR = DRIVE_ARTIFACT_DIR / "dataset"
DATASET_RELEASE_URL = "https://github.com/eritry/simpsons-classification/releases/download/dataset-v1/journey-springfield.zip"
DATASET_ARCHIVE_PATH = DATASET_DIR / "journey-springfield.zip"

BATCH_SIZE = 64

for directory in [DRIVE_ARTIFACT_DIR, CHECKPOINT_DIR, LABEL_AUDIT_DIR, DATASET_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# CSV with manually reviewed suspicious labels, versioned with the repository.
REVIEW_PATH = CODE_DIR / "artifacts" / "label_audit" / "suspicious_manual.csv"

## 3. Data Loading and Preprocessing

### Dataset Download and Cache

The dataset archive is hosted as a GitHub Release asset. On the first run, the notebook downloads it to Google Drive; later runs reuse the cached archive and only extract it into the current Colab runtime.

In [ ]:
from data_io import ensure_dataset

ensure_dataset(
    archive_url=DATASET_RELEASE_URL,
    archive_path=DATASET_ARCHIVE_PATH,
    extract_dir="/content",
    required_paths=[TRAIN_DIR, TEST_DIR, SAMPLE_SUBMISSION_PATH],
)

In [ ]:
# The dataset is now available under /content/train, /content/testset, and /content/sample_submission.csv.

In [ ]:
# Optional label-fix block based on the reviewed CSV committed to the repository.
# Enabled by default so training uses the reviewed labels before files are collected.
from label_audit import move_reviewed_files_to_class

APPLY_LABEL_FIXES = True

In [ ]:
if APPLY_LABEL_FIXES:
    if not REVIEW_PATH.exists():
        raise FileNotFoundError(f"Manual review file not found: {REVIEW_PATH}")

    review_df = pd.read_csv(REVIEW_PATH)
    moved_paths = move_reviewed_files_to_class(
        review_df=review_df,
        dry_run=False,
        dataset_root=TRAIN_DIR,
    )
else:
    moved_paths = {}
    print("Label fixing is disabled. Set APPLY_LABEL_FIXES = True to apply reviewed labels.")

In [ ]:
from dataset import collect_image_files

train_val_files, test_files = collect_image_files(TRAIN_DIR, TEST_DIR)
print("Total training/validation images:", len(train_val_files))
print("Test images:", len(test_files))

### Label Encoding

Class names are inferred from parent directory names. `LabelEncoder` maps these text labels to integer class IDs for training and later converts model predictions back to class names for the submission file.

In [ ]:
from dataset import create_label_encoder

label_encoder, train_val_labels = create_label_encoder(train_val_files)

### Train/Validation Split

The original training folder is split into training and validation subsets. The split is stratified by class label so that rare and frequent characters are represented proportionally in both subsets.

In [ ]:
from dataset import split_train_val_files

train_files, val_files = split_train_val_files(
    train_val_files=train_val_files,
    train_val_labels=train_val_labels,
    test_size=0.25,
    random_state=42,
)

### Dataset and DataLoader Objects

`SimpsonsDataset` applies stronger image augmentation during training and deterministic preprocessing for validation/test inference. This mirrors a standard production pattern: augment only the training signal, keep evaluation stable.

In [ ]:
from dataset import SimpsonsDataset, create_datasets

train_dataset, val_dataset = create_datasets(
    train_files=train_files,
    val_files=val_files,
    label_encoder=label_encoder,
)

In [ ]:
print("Train dataset examples:", len(train_dataset))
print("Validation dataset examples:", len(val_dataset))

### Class Distribution

The dataset is imbalanced: major characters have many more samples than rare characters. The plot below makes that imbalance explicit before training.

In [ ]:
from visualization import plot_train_val_distribution

distribution_df = plot_train_val_distribution(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    title="Train vs Validation distribution by Simpsons character",
)

### Imbalance Handling

A `WeightedRandomSampler` increases the probability of sampling under-represented classes. I use inverse square-root class frequency rather than full inverse frequency to reduce imbalance without making rare classes dominate every batch.

In [ ]:
from dataset import create_weighted_dataloaders

loaders, class_counts = create_weighted_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    label_encoder=label_encoder,
    batch_size=BATCH_SIZE,
    train_num_workers=2,
    val_num_workers=0,
)

train_loader = loaders["train"]
val_loader = loaders["val"]

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Number of classes:", len(label_encoder.classes_))

### Visual Sanity Check

Before training, it is useful to inspect augmented batches manually. This catches obvious preprocessing issues such as wrong labels, broken images, or overly aggressive transforms.

Visualization helpers are implemented in `visualization.py`. Keeping them outside the notebook makes the notebook easier to read and the helpers easier to reuse.

In [ ]:
from visualization import (
    show_images_from_loader,
    show_images_with_predictions,
)

In [ ]:
show_images_from_loader(n_rows=3, n_cols=4, loader=train_loader, label_encoder=label_encoder)

## 4. Model Architecture

### Model Candidates

This project compares three levels of model complexity:

- `SimpleCNN`: a small from-scratch baseline that establishes whether the pipeline learns at all.
- `DenseNet121`: the strongest final ImageNet-pretrained model by validation and Kaggle metrics.
- `EfficientNetV2-S`: the default walkthrough model because its training curves show the optimization process more clearly.

The notebook trains one selected model at a time to keep Colab runtime manageable. By default, it uses EfficientNetV2-S for the training-curve walkthrough and reports DenseNet121 as the best final reference model in the comparison table.

### Why Compare These Models

The comparison separates data-pipeline quality from architecture strength. If `SimpleCNN` works but underperforms, the pipeline is probably valid and transfer learning is the right next step. Comparing DenseNet121 and EfficientNetV2-S then shows how architecture choice, optimizer settings, and fine-tuning strategy affect macro F1 on an imbalanced dataset.

In [ ]:
# Choose one model for this notebook run.
# Options: "simple_cnn", "densenet121", "efficientnet_v2_s"
# EfficientNetV2-S is the default walkthrough model because its curves are easier to inspect.
MODEL_NAME = "efficientnet_v2_s"

TRANSFER_MODELS = {"densenet121", "efficientnet_v2_s"}

TRAINING_CONFIGS = {
    "simple_cnn": {
        "display_name": "SimpleCNN",
        "epochs": 15,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "label_smoothing": 0.0,
        "val_every_steps": None,
        "checkpoint_dir": CHECKPOINT_DIR / "simple_cnn",
        "best_checkpoint_dir": CHECKPOINT_DIR / "simple_cnn",
        "summary_dir": CHECKPOINT_DIR / "simple_cnn",
    },
    "densenet121": {
        "display_name": "DenseNet121",
        "strategy": "two_phase_no_freeze",
        "phase1_epochs": 5,
        "phase2_epochs": 5,
        "phase1_backbone_lr": 1e-4,
        "phase1_classifier_lr": 1e-3,
        "phase2_backbone_lr": 1e-6,
        "phase2_classifier_lr": 1e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.10,
        "eta_min": 1e-6,
        "val_every_steps": None,
        "phase1_dir": CHECKPOINT_DIR / "densenet121_phase1",
        "phase2_dir": CHECKPOINT_DIR / "densenet121_phase2",
        "best_checkpoint_dir": CHECKPOINT_DIR / "densenet121_phase2",
        "summary_dir": CHECKPOINT_DIR / "densenet121_phase2",
    },
    "efficientnet_v2_s": {
        "display_name": "EfficientNetV2-S",
        "head_epochs": 3,
        "fine_tune_epochs": 10,
        "head_lr": 1e-3,
        "backbone_lr": 1e-5,
        "classifier_lr": 3e-4,
        "weight_decay": 1e-4,
        "head_label_smoothing": 0.02,
        "fine_tune_label_smoothing": 0.02,
        "eta_min": 1e-6,
        "val_every_steps": None,
        "stage1_dir": CHECKPOINT_DIR / "efficientnet_v2_s_stage1",
        "stage2_dir": CHECKPOINT_DIR / "efficientnet_v2_s_stage2",
        "best_checkpoint_dir": CHECKPOINT_DIR / "efficientnet_v2_s_stage2",
        "summary_dir": CHECKPOINT_DIR / "efficientnet_v2_s_stage2",
    },
}

selected_config = TRAINING_CONFIGS[MODEL_NAME]

In [ ]:
from model import count_parameters, create_model

selected_model = create_model(
    model_name=MODEL_NAME,
    num_classes=len(label_encoder.classes_),
    dropout=0.3,
).to(DEVICE)

print("Selected model:", MODEL_NAME)
print("Total parameters:", f"{count_parameters(selected_model):,}")
print("Trainable parameters:", f"{count_parameters(selected_model, trainable_only=True):,}")
print("Checkpoint directory:", selected_config["best_checkpoint_dir"])

## 5. Training Utilities

Training, evaluation, checkpointing, plotting, and prediction helpers are imported from `training.py`. The notebook keeps experiment orchestration visible while moving reusable mechanics into focused modules.

In [ ]:
from training import (
    epoch_history_rows,
    plot_history,
    predict,
    summarize_history,
    train_baseline_model,
    train_transfer_model,
)

In [ ]:
history = None

## 6. Training and Validation

### Optimization Strategy

Training settings are declared in `TRAINING_CONFIGS` so model-specific choices are explicit and easy to compare. `SimpleCNN` trains all parameters from scratch with Adam. DenseNet121 uses two no-freeze fine-tuning phases with discriminative learning rates and reaches the strongest final metrics. EfficientNetV2-S keeps a more conservative schedule: train the classifier head first, then fine-tune the backbone with a lower learning rate.

The notebook defaults to EfficientNetV2-S for the main training walkthrough because its learning curves are more informative: the metrics improve gradually across the classifier-head and fine-tuning stages, making convergence and validation behavior easier to discuss. DenseNet121 remains important as the best final model, but its curve saturates quickly, which is expected for a strong pretrained backbone on this dataset.

The default training output is intentionally compact: one summary row per epoch, followed by `training_history_df` and `run_summary` tables. Step-level validation can still be enabled by setting `val_every_steps` in the selected config.

### Validation metric

The training loop tracks both accuracy and macro F1. Macro F1 is important here because the dataset is imbalanced: it gives rare classes the same weight as frequent classes when summarizing performance.

In [ ]:
# Class weights are used during transfer-model fine-tuning.
class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE,
)

In [ ]:
if MODEL_NAME == "simple_cnn":
    history = train_baseline_model(
        model=selected_model,
        dataloaders=loaders,
        device=DEVICE,
        config=selected_config,
    )
else:
    history = train_transfer_model(
        model=selected_model,
        model_name=MODEL_NAME,
        dataloaders=loaders,
        device=DEVICE,
        class_weights=class_weights,
        config=selected_config,
    )

plot_history(history)

training_history_df = pd.DataFrame(epoch_history_rows(history))
selected_config["summary_dir"].mkdir(parents=True, exist_ok=True)
training_history_df.to_csv(selected_config["summary_dir"] / "training_history.csv", index=False)

run_summary = pd.DataFrame([
    summarize_history(
        model_name=MODEL_NAME,
        history=history,
        parameter_count=count_parameters(selected_model),
    )
])

run_summary.to_csv(selected_config["summary_dir"] / "run_summary.csv", index=False)

display(training_history_df)
run_summary

In [ ]:
from sklearn.metrics import classification_report

selected_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        logits = selected_model(images)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

print(classification_report(
    all_targets,
    all_preds,
    target_names=label_encoder.classes_,
    digits=4
))

### Model Comparison

Representative results show the role of each model. `SimpleCNN` is a useful baseline but overfits: it reaches high train F1 while validation macro F1 saturates much lower. Pretrained models improve sample efficiency and rare-class performance.

| Model | Pretraining | Parameters | Validation Macro F1 | Validation Accuracy | Kaggle Score | Role |
|---|---:|---:|---:|---:|---:|---|
| SimpleCNN | No | 180,762 | 0.737 | 0.839 | not used as final | From-scratch baseline |
| DenseNet121 | ImageNet | 7.2M | 0.935 | 0.981 | 0.993 | Best final model |
| EfficientNetV2-S | ImageNet | 21M | 0.833 | 0.935 | 0.972 | Main training-curve walkthrough |

EfficientNetV2-S is used as the main walkthrough model because its learning curves make the training dynamics easier to inspect. DenseNet121 is kept as the best final model because it achieved the strongest validation macro F1, validation accuracy, and Kaggle score in this comparison using no-freeze fine-tuning with discriminative learning rates.

## 7. Error Analysis and Label Audit

After training, the model is used to find examples where it disagrees with the current folder label while being highly confident. These cases are good candidates for manual label review. This is not automatic relabeling: the CSV is meant to support human inspection.

In [ ]:
from label_audit import (
    run_label_audit,
    show_review_examples,
)

In [ ]:
review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_review.csv"
print("Label audit directory:", LABEL_AUDIT_DIR)

In [ ]:
audit_dataset, df_all, df_suspicious = run_label_audit(
    model=selected_model,
    train_files=train_val_files,
    label_encoder=label_encoder,
    dataset_cls=SimpsonsDataset,
    device=DEVICE,
    batch_size=64,
    num_workers=0,
    min_confidence=0.90,
    min_margin=0.20,
    top_n=300,
    save_all_csv_path=LABEL_AUDIT_DIR / "all_predictions.csv",
    save_review_csv_path=LABEL_AUDIT_DIR / "suspicious_labels_review.csv",
)

Label-audit helpers are implemented in `label_audit.py`. They scan predictions, rank suspicious examples, save review CSV files, and visualize candidates for manual inspection.

In [ ]:
review_df = df_suspicious.copy()
review_df["action"] = ""          # options: "move", "keep", "skip"
review_df["correct_label"] = ""   # for example: "homer_simpson"

review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_new.csv"
review_df.to_csv(review_csv_path, index=False)

show_review_examples(
    review_df=review_df,
    dataset=audit_dataset,
    start_pos=0,
    n_rows=4,
    n_cols=3,
)

### Prediction Confidence Visualization

The next visualization samples validation images and overlays the model's top prediction and confidence. This is a quick qualitative check of calibration and common confusions.

`show_images_with_predictions` is implemented in `visualization.py`.

In [ ]:
show_images_with_predictions(
    n_rows=3,
    n_cols=3,
    dataset=val_dataset,
    model=selected_model,
    label_encoder=label_encoder,
    device=DEVICE,
)

## 8. Kaggle-Style Submission

Create a deterministic test DataLoader. Test samples have no labels, so the dataset returns only transformed images.

In [ ]:
from dataset import create_test_loader

test_dataset, test_loader = create_test_loader(
    test_files=test_files,
    label_encoder=label_encoder,
    batch_size=BATCH_SIZE,
)

The `predict` helper returns integer class IDs for all test images.

Load the best checkpoint if it exists, then run inference on the test set.

In [ ]:
from submission import load_best_model_if_exists

best_checkpoint_path = selected_config["best_checkpoint_dir"] / "best_model.pth"

selected_model = load_best_model_if_exists(
    model=selected_model,
    checkpoint_path=best_checkpoint_path,
    device=DEVICE,
)

predicted_numeric_labels = predict(selected_model, test_loader, device=DEVICE)
predicted_numeric_labels = predicted_numeric_labels.numpy()

Convert numeric predictions back to text labels.

In [ ]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)

Load the sample submission format and create a final CSV with the required `Id` and `Expected` columns.

In [ ]:
from submission import create_submission

my_submission, sample_submission = create_submission(
    test_files=test_files,
    predicted_labels=predicted_text_labels,
    sample_submission_path=SAMPLE_SUBMISSION_PATH,
)

sample_submission.head(10)

In [ ]:
my_submission.head(10)

In [ ]:
# The CSV below can be uploaded to Kaggle or used for local review.

In [ ]:
my_submission.to_csv('simple_cnn_baseline.csv', index=False)